# Transformer — Attention Is All You Need

The transformer architecture powers every modern LLM: GPT, LLaMA, Mistral, BERT.
Unlike RNNs which process tokens one at a time, Transformers process the **entire sequence in parallel** using **self-attention**.

**Self-attention answers:** for each token in the sequence, which other tokens should I pay attention to?

We will build:
1. Scaled dot-product attention (the core operation)
2. Multi-head attention
3. A full Transformer block (as used in GPT)

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

# --- Scaled Dot-Product Attention ---
# Given Q (query), K (key), V (value) — all same shape [batch, seq, dim]
#
# attention(Q, K, V) = softmax(QK^T / sqrt(d_k)) * V
#
# QK^T  → similarity score between every pair of tokens
# /sqrt  → scale down to prevent vanishing gradients in softmax
# softmax → convert scores to weights that sum to 1
# * V     → weighted sum of values

def scaled_dot_product_attention(Q, K, V, mask=None):
    d_k    = Q.shape[-1]                          # dimension of each head
    scores = Q @ K.transpose(-2, -1) / math.sqrt(d_k)  # [batch, heads, seq, seq]

    if mask is not None:
        scores = scores.masked_fill(mask == 0, float('-inf'))  # causal mask for GPT

    weights = F.softmax(scores, dim=-1)           # attention weights
    return weights @ V, weights                   # context vectors, weights


# Test
batch, seq, d_model = 2, 5, 64
Q = torch.randn(batch, seq, d_model)
K = torch.randn(batch, seq, d_model)
V = torch.randn(batch, seq, d_model)

context, weights = scaled_dot_product_attention(Q, K, V)
print('Context shape:', context.shape)   # [2, 5, 64]
print('Weights shape:', weights.shape)   # [2, 5, 5] — attention matrix

In [ ]:
# --- Multi-Head Attention ---
# Instead of one big attention, run h smaller attentions in parallel ("heads")
# Each head can learn to attend to different aspects of the sequence
# (one head for syntax, another for semantics, etc.)

class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()
        assert d_model % num_heads == 0

        self.d_model    = d_model
        self.num_heads  = num_heads
        self.d_k        = d_model // num_heads   # dimension per head

        # Linear projections for Q, K, V and output
        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)

    def split_heads(self, x):
        # x: [batch, seq, d_model] → [batch, heads, seq, d_k]
        batch, seq, _ = x.shape
        x = x.reshape(batch, seq, self.num_heads, self.d_k)
        return x.transpose(1, 2)

    def forward(self, x, mask=None):
        Q = self.split_heads(self.W_q(x))
        K = self.split_heads(self.W_k(x))
        V = self.split_heads(self.W_v(x))

        context, _ = scaled_dot_product_attention(Q, K, V, mask)

        # Merge heads back: [batch, heads, seq, d_k] → [batch, seq, d_model]
        batch, _, seq, _ = context.shape
        context = context.transpose(1, 2).reshape(batch, seq, self.d_model)

        return self.W_o(context)


batch, seq, d_model = 2, 10, 128
mha = MultiHeadAttention(d_model=128, num_heads=4)
x   = torch.randn(batch, seq, d_model)
out = mha(x)
print('MHA output shape:', out.shape)   # [2, 10, 128]

In [ ]:
# --- Full Transformer Block (GPT-style) ---
# Each block: LayerNorm → MHA → residual → LayerNorm → FFN → residual
# The residual connections (x + sublayer(x)) are critical — they allow gradients to flow

class TransformerBlock(nn.Module):
    def __init__(self, d_model, num_heads, ffn_dim, dropout=0.1):
        super().__init__()
        self.attn    = MultiHeadAttention(d_model, num_heads)
        self.ffn     = nn.Sequential(
            nn.Linear(d_model, ffn_dim),
            nn.GELU(),                     # GELU is standard in modern LLMs
            nn.Linear(ffn_dim, d_model)
        )
        self.norm1   = nn.LayerNorm(d_model)
        self.norm2   = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, mask=None):
        # Pre-norm architecture (used in GPT-2, LLaMA) — more stable than post-norm
        x = x + self.dropout(self.attn(self.norm1(x), mask))   # attention + residual
        x = x + self.dropout(self.ffn(self.norm2(x)))           # FFN + residual
        return x


block = TransformerBlock(d_model=128, num_heads=4, ffn_dim=512)
x     = torch.randn(2, 10, 128)
out   = block(x)
print('Transformer block output shape:', out.shape)   # [2, 10, 128]

## Three Transformer Families — which one is "an LLM"?

The transformer block you just built is a universal component. How you *stack and configure* it determines what kind of model you get.

```
┌─────────────────────────────────────────────────────────────────────┐
│  ENCODER-ONLY          ENCODER-DECODER         DECODER-ONLY         │
│  (BERT-style)          (T5-style)               (GPT-style)         │
│                                                                      │
│  Token → [ENC] → vec  Prompt → [ENC] → ctx    Token → [DEC] → next │
│                        ctx + output → [DEC]                         │
│                                                                      │
│  Sees ALL tokens       Encoder: sees all        Only sees PAST       │
│  at once (no mask)     Decoder: causal mask     tokens (causal mask) │
│                                                                      │
│  BERT, RoBERTa         T5, BART, mT5            GPT-4, Claude,      │
│                                                  LLaMA, Mistral      │
│  Use for:              Use for:                 Use for:             │
│  classification,       translation,             generation,          │
│  embeddings,           summarization            chat, code,          │
│  reward models         (seq2seq tasks)          post-training        │
└─────────────────────────────────────────────────────────────────────┘
```

**The difference is the mask — that's it.**

- Encoder: **no causal mask** → every token attends to every other token (bidirectional)
- Decoder: **causal mask** → each token only attends to past tokens (left-to-right)

The MiniGPT below is a decoder-only model — the same family as LLaMA and Mistral.
This is what you will fine-tune in Level 5 and post-train in your startup platform.

**Why decoder-only won:** encoder-decoder and encoder-only models need different training objectives (masked language modeling, span corruption). Decoder-only with next-token prediction scales better and is simpler to train at massive scale. GPT-3 proved this in 2020 — the field moved to decoder-only.

In [ ]:
# --- Minimal GPT — stacked Transformer blocks ---

class MiniGPT(nn.Module):
    def __init__(self, vocab_size, d_model, num_heads, num_layers, max_seq_len, ffn_dim):
        super().__init__()
        self.token_embed = nn.Embedding(vocab_size, d_model)
        self.pos_embed   = nn.Embedding(max_seq_len, d_model)  # learned positional embeddings
        self.blocks      = nn.ModuleList([
            TransformerBlock(d_model, num_heads, ffn_dim) for _ in range(num_layers)
        ])
        self.norm        = nn.LayerNorm(d_model)
        self.head        = nn.Linear(d_model, vocab_size)      # project back to vocab

    def forward(self, token_ids):
        batch, seq_len = token_ids.shape
        positions = torch.arange(seq_len, device=token_ids.device).unsqueeze(0)  # [1, seq]

        x = self.token_embed(token_ids) + self.pos_embed(positions)  # token + position

        # Causal mask — each token can only attend to previous tokens (autoregressive)
        mask = torch.tril(torch.ones(seq_len, seq_len, device=token_ids.device)).unsqueeze(0).unsqueeze(0)

        for block in self.blocks:
            x = block(x, mask)

        x      = self.norm(x)
        logits = self.head(x)   # [batch, seq, vocab_size]
        return logits


gpt = MiniGPT(vocab_size=1000, d_model=128, num_heads=4, num_layers=4, max_seq_len=64, ffn_dim=512)
tokens = torch.randint(0, 1000, (2, 16))   # [batch=2, seq=16]
logits = gpt(tokens)

total = sum(p.numel() for p in gpt.parameters())
print(f'MiniGPT parameters: {total:,}')
print(f'Output logits shape: {logits.shape}')   # [2, 16, 1000]

## Autoregressive Generation — how GPT actually produces text

Training feeds the model a full sequence and computes loss on all positions at once (teacher forcing).

**Inference is completely different.** The model has no future tokens — it must generate them one at a time:

```
Step 1: model(["The"])          → logits for next token → pick "cat"
Step 2: model(["The", "cat"])   → logits for next token → pick "sat"
Step 3: model(["The", "cat", "sat"]) → ...
```

Each new token is **appended to the input and fed back in**. This is *autoregressive* generation.

The causal mask you built above is what makes this safe — token N can only attend to tokens 0…N-1, so the model naturally predicts "what comes next" at every position.

Two ways to pick the next token from logits:
- **Greedy**: always take `argmax` — deterministic, often repetitive
- **Sampling**: draw from the probability distribution — creative, controlled by `temperature`
  - `temperature < 1.0` → sharper distribution, more focused
  - `temperature > 1.0` → flatter distribution, more random

In [ ]:
@torch.no_grad()
def generate(model, prompt_ids, max_new_tokens=20, temperature=1.0):
    """
    Autoregressive generation loop — the core of how every GPT-style model runs at inference.

    prompt_ids: [1, prompt_len]  — starting context (token IDs)
    Each iteration:
      1. Forward the FULL sequence so far through the model
      2. Take logits ONLY at the last position (that's our next-token prediction)
      3. Sample or pick argmax to get next token
      4. Append next token to sequence → feed back in next iteration
    """
    model.eval()
    tokens = prompt_ids   # [1, seq_len]

    for _ in range(max_new_tokens):
        logits     = model(tokens)          # [1, seq_len, vocab_size]
        next_logits = logits[:, -1, :]      # [1, vocab_size] — only the LAST position matters

        if temperature == 0.0:
            # Greedy decoding — always pick the most likely token
            next_token = next_logits.argmax(dim=-1, keepdim=True)   # [1, 1]
        else:
            # Sampling with temperature
            probs      = torch.softmax(next_logits / temperature, dim=-1)
            next_token = torch.multinomial(probs, num_samples=1)     # [1, 1]

        # Append and feed back — this is what "autoregressive" means
        tokens = torch.cat([tokens, next_token], dim=1)   # [1, seq_len + 1]

    return tokens


# Untrained model — output is random token IDs, but the LOOP is correct
# This exact pattern is what model.generate() in HuggingFace does internally
prompt    = torch.randint(0, 1000, (1, 5))   # batch=1, 5 prompt tokens
generated = generate(gpt, prompt, max_new_tokens=10, temperature=1.0)

print(f'Prompt ({prompt.shape[1]} tokens):    {prompt[0].tolist()}')
print(f'Generated ({generated.shape[1]} tokens): {generated[0].tolist()}')
print(f'New tokens added: {generated.shape[1] - prompt.shape[1]}')

# Show how sequence grows at each step
print('\nHow the sequence grows:')
tokens = prompt.clone()
for step in range(5):
    logits     = gpt(tokens)
    next_token = logits[:, -1, :].argmax(dim=-1, keepdim=True)
    tokens     = torch.cat([tokens, next_token], dim=1)
    print(f'  Step {step+1}: length={tokens.shape[1]}  last token={next_token.item()}')

## Sampling Strategies — controlling what the model generates

Temperature alone gives you a dial between "boring" and "random."
But production LLMs use smarter strategies to filter the distribution before sampling.

```
Raw logits: [cat=5.2, dog=4.8, sat=3.1, xyz=0.2, aaa=0.1, ...]  (50,000 tokens)

Greedy:     always pick "cat" → repetitive, boring
Temperature: scale logits → sample → might pick "xyz" → incoherent

Top-k:      keep only the k most likely tokens, zero out the rest → sample from those
Top-p:      keep tokens until cumulative probability reaches p → sample from those
```

**Top-k** = fixed number of candidates (e.g., top 50 tokens)
**Top-p** (nucleus sampling) = dynamic number — keeps adding tokens until probability mass reaches p (e.g., 0.9)

Top-p is generally better because it adapts: confident predictions → few candidates, uncertain → many.

In [ ]:
def top_k_sampling(logits, k=50):
    """Keep only the top-k tokens, zero out the rest."""
    values, indices = logits.topk(k)                      # [1, k]
    filtered = torch.full_like(logits, float('-inf'))      # start with all -inf
    filtered.scatter_(1, indices, values)                  # put top-k values back
    return filtered


def top_p_sampling(logits, p=0.9):
    """
    Nucleus sampling — keep tokens until cumulative probability >= p.
    Adapts dynamically: confident predictions → fewer candidates, uncertain → more.
    """
    sorted_logits, sorted_indices = logits.sort(descending=True)
    cumulative_probs = torch.softmax(sorted_logits, dim=-1).cumsum(dim=-1)

    # Find where cumulative probability first exceeds p
    mask = cumulative_probs - torch.softmax(sorted_logits, dim=-1) >= p   # shift by one so we INCLUDE the token that crosses p
    sorted_logits[mask] = float('-inf')

    # Unsort back to original order
    filtered = sorted_logits.gather(1, sorted_indices.argsort(1))
    return filtered


@torch.no_grad()
def generate_with_sampling(model, prompt_ids, max_new_tokens=30,
                           temperature=0.8, top_k=50, top_p=0.9, strategy='top_p'):
    """
    Production-style generation with all three strategies.
    This is what model.generate() in HuggingFace does under the hood.
    """
    model.eval()
    tokens = prompt_ids

    for _ in range(max_new_tokens):
        logits      = model(tokens)
        next_logits = logits[:, -1, :]          # [1, vocab_size]

        # Step 1: temperature scaling
        next_logits = next_logits / temperature

        # Step 2: filter distribution
        if strategy == 'greedy':
            next_token = next_logits.argmax(dim=-1, keepdim=True)
            tokens = torch.cat([tokens, next_token], dim=1)
            continue
        elif strategy == 'top_k':
            next_logits = top_k_sampling(next_logits, k=top_k)
        elif strategy == 'top_p':
            next_logits = top_p_sampling(next_logits, p=top_p)

        # Step 3: sample from filtered distribution
        probs      = torch.softmax(next_logits, dim=-1)
        next_token = torch.multinomial(probs, num_samples=1)
        tokens     = torch.cat([tokens, next_token], dim=1)

    return tokens


# Compare strategies — same prompt, different behaviors
prompt = torch.randint(0, 1000, (1, 5))

print('=== Sampling Strategy Comparison ===\n')
for strategy in ['greedy', 'top_k', 'top_p']:
    out = generate_with_sampling(gpt, prompt, max_new_tokens=10, strategy=strategy)
    new_tokens = out[0, prompt.shape[1]:].tolist()
    print(f'{strategy:8s}: {new_tokens}')

# Show how top-p adapts: count how many tokens survive filtering
logits_example = torch.randn(1, 1000)  # fake logits
for p in [0.5, 0.9, 0.95]:
    filtered = top_p_sampling(logits_example.clone(), p=p)
    surviving = (filtered > float('-inf')).sum().item()
    print(f'\ntop_p={p}: {surviving} tokens survive out of 1000')